# STAGE I — Seed Ensemble (고정 튜닝 파라미터)

> 캐글 단독으로 처음부터 끝까지 실행 가능합니다.

**배경**: STAGE E에서 trial을 8개에서 60개로 늘렸더니(FAST=False, 본 튜닝)
**OOF는 올랐지만(0.74039에서 0.74059) LB는 오히려 떨어졌습니다(0.74187에서
0.74182)**. 이건 정교한 탐색일수록 탐색에 쓰인 특정 fold 구성에 과적합될
위험이 커진다는 신호로 보입니다.

**이번 접근**: 탐색을 더 정교하게 하는 대신, 이미 찾은 파라미터의 안정성을
높이는 방향으로 갑니다. 같은 하이퍼파라미터로 fold 분할만 다른 시드로
5번 반복해서 평균을 내면, 어떤 특정 fold 구성에서 우연히 잘 맞았다는
효과가 줄어들고 더 일반화된 예측에 가까워질 수 있습니다.

**사용하는 파라미터**: STAGE E(FAST=False)에서 찾은 best params를 그대로
고정해서 씁니다(재탐색하지 않습니다).

**비교 기준**:
- 단일 seed(42) 결과: OOF 0.74059, LB 0.74182
- 5-seed ensemble 결과: 이번에 확인
- 참고로 LB 최고 기록: 0.74187 (STAGE E FAST=True)

**비용 가이드**: 5-fold x 4모델 x 5seed = 100번의 fold 학습. STAGE E의
5-fold 재학습 1회가 몇 분에서 수십 분 걸렸다면, 이번엔 그 5배입니다.
GPU 켜고 진행하시는 걸 권장합니다.


## 1. 셋업

In [ ]:
!pip install -q koreanize-matplotlib


In [ ]:
import re, os, glob, warnings, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)


def kfont():
    try:
        import koreanize_matplotlib; return "koreanize_matplotlib"
    except Exception:
        pass
    from matplotlib import font_manager
    for c in ["NanumGothic", "Malgun Gothic", "AppleGothic"]:
        if any(c in f.name for f in font_manager.fontManager.ttflist):
            matplotlib.rcParams["font.family"] = c
            return c
    matplotlib.rcParams["axes.unicode_minus"] = False
    return None


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("폰트:", kfont() or "미설정", "| lgb", lgb.__version__, "| xgb", xgb.__version__, "| DEVICE:", DEVICE)


## 2. 데이터 로드 (캐글 경로)

In [ ]:
DATA_DIR = None  # 자동 탐색 안 되면 여기에 직접 지정 (예: "/kaggle/input/datasets/tnsgusm/tnsgusm12")

if DATA_DIR:
    csvs = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
else:
    csvs = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))


def pick(*ks):
    for p in csvs:
        if all(k in os.path.basename(p).lower() for k in ks):
            return p


train_path = pick("train")
test_path = pick("test")
sub_path = pick("submission") or pick("sample")
print("train:", train_path)
print("test :", test_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sub_path) if sub_path else None
print("train", train.shape, "test", test.shape)


In [ ]:
ID_COL = "ID"
N_SPLITS = 5
BASE_SEED = 42
SEEDS = [42, 7, 100, 777, 2024]

lgb_params = {'learning_rate': 0.01668577399627212, 'num_leaves': 108, 'max_depth': 4,
              'min_child_samples': 72, 'feature_fraction': 0.9986147780453636,
              'bagging_fraction': 0.8137099133074838, 'reg_alpha': 9.921763896126146,
              'reg_lambda': 0.9116566354544959}
cat_params = {'learning_rate': 0.015434354584269611, 'depth': 7, 'l2_leaf_reg': 5.575950520858999}
xgb_params = {'learning_rate': 0.017233564711968248, 'max_depth': 5, 'min_child_weight': 19,
              'subsample': 0.6871745250676744, 'colsample_bytree': 0.9944913750500002,
              'reg_alpha': 9.780488200512217, 'reg_lambda': 0.04980498927284957}
mlp_params = {'n_layers': 3, 'hidden_0': 178, 'hidden_1': 77, 'hidden_2': 126,
              'dropout': 0.40563409410211304, 'lr': 0.001181848547109624}

print("고정 파라미터 로드 완료")
print("SEEDS:", SEEDS)


## 3. 컬럼/매핑 정의 + 전처리 (STAGE D/E와 동일)

In [ ]:
TARGET = "임신 성공 여부"; CLINIC_COL = "시술 시기 코드"
COL_PROC = "특정 시술 유형"; COL_RSN = "배아 생성 주요 이유"
ORDINAL_COUNT_COLS = ["총 시술 횟수", "클리닉 내 총 시술 횟수", "IVF 시술 횟수", "DI 시술 횟수",
                       "총 임신 횟수", "IVF 임신 횟수", "DI 임신 횟수", "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수"]
NOMINAL_COLS = ["시술 시기 코드", "시술 유형", "배란 유도 유형", "난자 출처", "정자 출처"]
COUNT_MAP = {"0회": 0, "1회": 1, "2회": 2, "3회": 3, "4회": 4, "5회": 5, "6회 이상": 6}
AGE_T = {"만18-34세": 0, "만35-37세": 1, "만38-39세": 2, "만40-42세": 3, "만43-44세": 4, "만45-50세": 5, "알 수 없음": -1}
AGE_D = {"만20세 이하": 0, "만21-25세": 1, "만26-30세": 2, "만31-35세": 3, "만36-40세": 4, "만41-45세": 5, "알 수 없음": -1}
AGE_MAPS = {"시술 당시 나이": AGE_T, "난자 기증자 나이": AGE_D, "정자 기증자 나이": AGE_D}
_tp = lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]", str(s)) if t.strip()]
_tr2 = lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[,/]", str(s)) if t.strip()]


def fit_pre(tr):
    st = {}
    ig = {TARGET, ID_COL}
    st["dead"] = [c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True) <= 1]
    st["sparse"] = [c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean() > 0.98]
    nom = [c for c in NOMINAL_COLS if c in tr.columns]
    st["nominal"] = nom
    st["label_cats"] = {c: pd.Index(tr[c].astype("category").cat.categories) for c in nom}
    st["proc_vocab"] = sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}) if COL_PROC in tr else []
    st["reason_vocab"] = sorted({t for L in tr[COL_RSN].apply(_tr2) for t in L}) if COL_RSN in tr else []
    return st


def _common_drops(df, st):
    df = df.copy()
    if TARGET in df:
        df = df.drop(columns=[TARGET])
    if "시술 유형" in df:
        df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df = df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df:
            df[f"{c}_있음"] = df[c].notna().astype(int)
            df = df.drop(columns=[c])
    for c in ORDINAL_COUNT_COLS:
        if c in df:
            df[c] = df[c].map(COUNT_MAP)
    for c, m in AGE_MAPS.items():
        if c in df:
            df[c] = df[c].map(m)
    if COL_PROC in df:
        ts = df[COL_PROC].apply(_tp)
        for v in st["proc_vocab"]:
            df[f"proc_{v}"] = ts.apply(lambda L, v=v: int(v in L))
        df = df.drop(columns=[COL_PROC])
    if COL_RSN in df:
        ts = df[COL_RSN].apply(_tr2)
        for v in st["reason_vocab"]:
            df[f"rsn_{v}"] = ts.apply(lambda L, v=v: int(v in L))
        df = df.drop(columns=[COL_RSN])
    df = df.drop(columns=[c for c in [ID_COL] if c in df.columns])
    return df


def transform(df, st):
    df = _common_drops(df, st)
    cats = []
    for c, cc in st["label_cats"].items():
        if c in df:
            df[c] = pd.Categorical(df[c], categories=cc).codes.astype("int32")
            cats.append(c)
    obj = [c for c in df.columns if df[c].dtype == object]
    if obj:
        df = df.drop(columns=obj)
    return df, [c for c in cats if c in df.columns]


def transform_catboost(df, st):
    df = _common_drops(df, st)
    cats = []
    for c in st["nominal"]:
        if c in df:
            df[c] = df[c].astype(str).where(df[c].notna(), None)
            cats.append(c)
    return df, cats


def make_xy(tr, te):
    st = fit_pre(tr)
    Xtr, cats = transform(tr, st)
    Xte, _ = transform(te, st)
    Xte = Xte.reindex(columns=Xtr.columns)
    for c in cats:
        Xtr[c] = Xtr[c].fillna(-1).astype("int32")
        Xte[c] = Xte[c].fillna(-1).astype("int32")

    Xtr_cb, cats_cb = transform_catboost(tr, st)
    Xte_cb, _ = transform_catboost(te, st)
    Xte_cb = Xte_cb.reindex(columns=Xtr_cb.columns)

    y = tr[TARGET].astype(int).values
    ids = te[ID_COL].values
    return Xtr, y, Xte, ids, [c for c in cats if c in Xtr.columns], Xtr_cb, Xte_cb, cats_cb


Xtr, y, Xte, ids, cats, Xtr_cb, Xte_cb, cats_cb = make_xy(train, test)
print("Xtr", Xtr.shape, "Xtr_cb", Xtr_cb.shape, "범주형", cats)


## 4. 데이터 누수 자동 검증

In [ ]:
from sklearn.model_selection import cross_val_score

av_X = pd.concat([Xtr.assign(_is_test=0), Xte.assign(_is_test=1)], ignore_index=True)
av_y = av_X.pop("_is_test").values
av_model = lgb.LGBMClassifier(n_estimators=200, max_depth=4, verbose=-1, random_state=BASE_SEED)
av_auc = cross_val_score(av_model, av_X, av_y, cv=3, scoring="roc_auc").mean()
print(f"Adversarial Validation AUC = {av_auc:.4f}  (0.5에 가까울수록 정상)")


## 5. MLP 정의 (STAGE D/E와 동일)

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def prepare_mlp_input(Xtr_df, Xte_df, cat_cols, fit_on_train=True, scaler=None, ohe_cats=None):
    num_cols = [c for c in Xtr_df.columns if c not in cat_cols]
    if fit_on_train:
        scaler = StandardScaler().fit(Xtr_df[num_cols].fillna(0))
        ohe_cats = {c: sorted(Xtr_df[c].dropna().unique().tolist()) for c in cat_cols}

    def _build(df):
        num_part = scaler.transform(df[num_cols].fillna(0))
        oh_parts = []
        for c in cat_cols:
            cats_list = ohe_cats[c]
            idx = {v: i for i, v in enumerate(cats_list)}
            oh = np.zeros((len(df), len(cats_list)), dtype=np.float32)
            codes = df[c].map(idx)
            valid = codes.notna()
            oh[np.arange(len(df))[valid], codes[valid].astype(int)] = 1.0
            oh_parts.append(oh)
        parts = [num_part.astype(np.float32)] + oh_parts
        return np.concatenate(parts, axis=1)

    Xtr_arr = _build(Xtr_df)
    Xte_arr = _build(Xte_df) if Xte_df is not None else None
    return Xtr_arr, Xte_arr, scaler, ohe_cats


def train_mlp_fold(Xt_arr, yt, Xv_arr, yv, Xte_arr, hidden, dropout, lr, epochs=40,
                    batch_size=2048, seed=42):
    torch.manual_seed(seed)
    in_dim = Xt_arr.shape[1]
    model = SimpleMLP(in_dim, hidden=hidden, dropout=dropout).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    pos_weight = torch.tensor([(yt == 0).sum() / max((yt == 1).sum(), 1)], dtype=torch.float32).to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    ds = TensorDataset(torch.tensor(Xt_arr, dtype=torch.float32), torch.tensor(yt, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    Xv_t = torch.tensor(Xv_arr, dtype=torch.float32).to(DEVICE)

    best_auc, best_state, patience, bad = -1, None, 8, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_logits = model(Xv_t).cpu().numpy()
        val_auc = roc_auc_score(yv, val_logits)
        if val_auc > best_auc:
            best_auc, best_state, bad = val_auc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = torch.sigmoid(model(Xv_t)).cpu().numpy()
        test_pred = torch.sigmoid(model(torch.tensor(Xte_arr, dtype=torch.float32).to(DEVICE))).cpu().numpy() \
            if Xte_arr is not None else None
    return val_pred, test_pred, best_auc

print("MLP 함수 정의 완료")


## 6. 고정 파라미터로 5-fold 학습 함수 (seed별로 반복 호출)

In [ ]:
def train_four_fixed(X, y, Xte, cats, X_cb, Xte_cb, cats_cb, n_splits, seed,
                      lgb_p, cat_p, xgb_p, mlp_p, verbose=True):
    folds = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(X, y))
    names = ["lgb", "cat", "xgb", "mlp"]
    oof = {m: np.zeros(len(X)) for m in names}
    tst = {m: np.zeros(len(Xte)) for m in names}
    nr, stop = 3000, 100

    n_layers = mlp_p.get("n_layers", 2)
    hidden = tuple(mlp_p[f"hidden_{i}"] for i in range(n_layers))
    cat_task_type = "GPU" if torch.cuda.is_available() else "CPU"
    xgb_device = "cuda" if torch.cuda.is_available() else "cpu"

    for k, (tri, vai) in enumerate(folds):
        Xt, Xv, yt, yv = X.iloc[tri], X.iloc[vai], y[tri], y[vai]
        Xt_cb, Xv_cb = X_cb.iloc[tri], X_cb.iloc[vai]

        dtr = lgb.Dataset(Xt, yt, categorical_feature=cats or "auto")
        dva = lgb.Dataset(Xv, yv, reference=dtr)
        lm = lgb.train(dict(objective="binary", metric="auc", verbose=-1, seed=seed, **lgb_p),
                        dtr, num_boost_round=nr, valid_sets=[dva],
                        callbacks=[lgb.early_stopping(stop, verbose=False), lgb.log_evaluation(0)])
        oof["lgb"][vai] = lm.predict(Xv)
        tst["lgb"] += lm.predict(Xte) / n_splits

        cm = CatBoostClassifier(iterations=nr, eval_metric="AUC", random_seed=seed, verbose=0,
                                 task_type=cat_task_type, early_stopping_rounds=stop, **cat_p)
        cm.fit(Pool(Xt_cb, yt, cat_features=cats_cb), eval_set=Pool(Xv_cb, yv, cat_features=cats_cb))
        oof["cat"][vai] = cm.predict_proba(Xv_cb)[:, 1]
        tst["cat"] += cm.predict_proba(Xte_cb)[:, 1] / n_splits

        xm = xgb.XGBClassifier(n_estimators=nr, tree_method="hist", device=xgb_device, eval_metric="auc",
                                random_state=seed, early_stopping_rounds=stop, **xgb_p)
        xm.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)
        oof["xgb"][vai] = xm.predict_proba(Xv)[:, 1]
        tst["xgb"] += xm.predict_proba(Xte)[:, 1] / n_splits

        Xt_arr, _, scaler, ohe_cats = prepare_mlp_input(Xt, None, cats, fit_on_train=True)
        _, Xv_arr, _, _ = prepare_mlp_input(Xt, Xv, cats, fit_on_train=False, scaler=scaler, ohe_cats=ohe_cats)
        _, Xte_arr, _, _ = prepare_mlp_input(Xt, Xte, cats, fit_on_train=False, scaler=scaler, ohe_cats=ohe_cats)
        val_pred, test_pred, mlp_auc = train_mlp_fold(
            Xt_arr, yt, Xv_arr, yv, Xte_arr,
            hidden=hidden, dropout=mlp_p.get("dropout", 0.3), lr=mlp_p.get("lr", 1e-3),
            epochs=40, seed=seed,
        )
        oof["mlp"][vai] = val_pred
        tst["mlp"] += test_pred / n_splits

        if verbose:
            print(f"    fold{k}  lgb={roc_auc_score(yv, oof['lgb'][vai]):.5f}"
                  f"  cat={roc_auc_score(yv, oof['cat'][vai]):.5f}"
                  f"  xgb={roc_auc_score(yv, oof['xgb'][vai]):.5f}"
                  f"  mlp={mlp_auc:.5f}")

    return oof, tst


def hill_climb(oof, y, n_iter=150):
    names = list(oof)
    single_ = {n: roc_auc_score(y, oof[n]) for n in names}
    best0 = max(single_, key=single_.get)
    ens = [best0]
    sums = oof[best0].copy()
    snaps = [(list(ens), roc_auc_score(y, sums / len(ens)))]
    for _ in range(n_iter):
        cb, ca = None, -1
        for n in names:
            a = roc_auc_score(y, (sums + oof[n]) / (len(ens) + 1))
            if a > ca:
                ca, cb = a, n
        ens.append(cb)
        sums = sums + oof[cb]
        snaps.append((list(ens), roc_auc_score(y, sums / len(ens))))
    best_ens, best_auc = max(snaps, key=lambda t: t[1])
    from collections import Counter
    c = Counter(best_ens)
    return {n: c.get(n, 0) / len(best_ens) for n in names}, best_auc, best0, single_

print("학습 함수 정의 완료")


## 7. 5개 seed로 반복 학습

각 seed마다 4모델 학습 후 힐클라이밍 블렌드를 만들고, 그 블렌드 OOF/test
예측을 저장합니다. 마지막에 5개 seed의 블렌드 예측을 다시 평균내는 것이
최종 seed ensemble입니다.

In [ ]:
seed_blend_oof = {}
seed_blend_tst = {}
seed_blend_auc = {}

for seed in SEEDS:
    print(f"\n=== Seed {seed} 학습 시작 ===")
    set_global_seed(seed)
    oof_s, tst_s = train_four_fixed(
        Xtr, y, Xte, cats, Xtr_cb, Xte_cb, cats_cb, N_SPLITS, seed,
        lgb_params, cat_params, xgb_params, mlp_params, verbose=True,
    )
    w_s, blend_auc_s, best0_s, single_s = hill_climb(oof_s, y)
    blend_oof_s = sum(w_s[m] * oof_s[m] for m in w_s)
    blend_tst_s = sum(w_s[m] * tst_s[m] for m in w_s)

    seed_blend_oof[seed] = blend_oof_s
    seed_blend_tst[seed] = blend_tst_s
    seed_blend_auc[seed] = blend_auc_s

    w_round = {k: round(v, 3) for k, v in w_s.items()}
    print(f"  Seed {seed} 블렌드 OOF AUC = {blend_auc_s:.5f}  (가중치: {w_round})")

    pd.DataFrame({"ID": train[ID_COL].values, "y": y,
                  **{f"oof_seed{s}": seed_blend_oof[s] for s in seed_blend_oof}}).to_csv(
        "/kaggle/working/seed_ensemble_oof_partial.csv", index=False)
    pd.DataFrame({"ID": ids, **{f"test_seed{s}": seed_blend_tst[s] for s in seed_blend_tst}}).to_csv(
        "/kaggle/working/seed_ensemble_test_partial.csv", index=False)
    print(f"  중간 저장 완료 (지금까지 {len(seed_blend_oof)}/{len(SEEDS)} seed)")


## 8. Seed Ensemble 결과 + 비교

In [ ]:
print("=== 개별 seed 블렌드 OOF AUC ===")
for s in SEEDS:
    print(f"  seed={s}: {seed_blend_auc[s]:.5f}")

seed_mean = np.mean(list(seed_blend_auc.values()))
seed_std = np.std(list(seed_blend_auc.values()))
print(f"\n개별 seed 평균: {seed_mean:.5f}  (표준편차 {seed_std:.5f})")

ensemble_oof = np.mean([seed_blend_oof[s] for s in SEEDS], axis=0)
ensemble_test = np.mean([seed_blend_tst[s] for s in SEEDS], axis=0)
ensemble_auc = roc_auc_score(y, ensemble_oof)

print(f"\nSeed Ensemble(5개 평균) OOF AUC = {ensemble_auc:.5f}")
print(f"  개별 seed 평균({seed_mean:.5f}) 대비 Δ = {ensemble_auc - seed_mean:+.5f}")
print(f"\n참고: STAGE E 단일 seed(42) 결과 = OOF 0.74059, LB 0.74182")
print(f"      STAGE E FAST=True 결과       = OOF 0.74039, LB 0.74187 (현재 LB 최고)")

if ensemble_auc > 0.74059 + 0.0002:
    print("\n결론: Seed ensemble이 의미있게 더 좋습니다. LB로 확인해볼 가치가 있습니다.")
elif ensemble_auc > 0.74059:
    print("\n결론: Seed ensemble이 약간 더 좋지만 작은 차이입니다. LB로 확인 권장.")
else:
    print("\n결론: Seed ensemble이 단일 seed보다 나아지지 않았습니다.")


## 9. 시각화: seed별 OOF AUC 분산

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
xs = list(range(len(SEEDS)))
vals = [seed_blend_auc[s] for s in SEEDS]
ax.scatter(xs, vals, s=80, label="개별 seed", zorder=3)
ax.axhline(seed_mean, color="gray", linestyle="--", label=f"개별 평균={seed_mean:.5f}")
ax.axhline(ensemble_auc, color="red", linestyle="-", label=f"Seed Ensemble={ensemble_auc:.5f}")
ax.set_xticks(xs)
ax.set_xticklabels([str(s) for s in SEEDS])
ax.set_xlabel("Seed")
ax.set_ylabel("블렌드 OOF AUC")
ax.set_title("Seed별 분산 vs Seed Ensemble")
ax.legend()
plt.tight_layout()
plt.show()


## 10. 제출 파일 (검증 포함)

In [ ]:
def write_sub(pred, name):
    pred = np.asarray(pred, dtype=float)
    issues = []
    if np.isnan(pred).any():
        issues.append(f"NaN {int(np.isnan(pred).sum())}개 포함")
    if (pred < 0).any() or (pred > 1).any():
        issues.append(f"[0,1] 범위 밖 값 존재 (min={pred.min():.4f}, max={pred.max():.4f})")
    if pred.std() < 1e-6:
        issues.append("예측값이 거의 상수입니다")

    s = sample_sub.copy() if sample_sub is not None else pd.DataFrame({ID_COL: ids})
    pc = [c for c in s.columns if c != ID_COL]
    pc = pc[0] if pc else "probability"

    if ID_COL in s.columns:
        if len(s) != len(ids) or set(s[ID_COL]) != set(ids):
            issues.append(f"sample_submission 행 수/ID 불일치 (sample={len(s)}개, 예측={len(ids)}개)")
        pm = dict(zip(ids, pred))
        s[pc] = s[ID_COL].map(pm)
        if s[pc].isna().any():
            issues.append(f"매핑 후 결측 {int(s[pc].isna().sum())}개")
    else:
        s[ID_COL] = ids
        s[pc] = pred

    if issues:
        print(f"문제 발견 [{name}]:")
        for it in issues:
            print(f"  - {it}")
    else:
        print(f"제출 파일 점검 통과 [{name}]")

    s.to_csv(name, index=False)
    return s, pc


sub, pc = write_sub(ensemble_test, "/kaggle/working/submission_seed_ensemble.csv")
print(f"\nsubmission_seed_ensemble.csv 저장: {sub.shape}")
sub.head()
